# 01 — Ingestion smoke test

Goal: verify that `tsr ingest` populated `Documents` + `TranscriptSegments` for our tracked creators.

This notebook is **read-only**. It imports from `app.*` and does not modify the database.

Run order:
1. `tsr initdb`
2. `tsr ingest --limit 5`
3. Open this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sqlalchemy import select

from app.db import session_scope
from app.models import Creator, Document, SourceChannel, TranscriptSegment

## Creators tracked

In [ ]:
with session_scope() as s:
    rows = s.execute(
        select(Creator.display_name, SourceChannel.source_type, SourceChannel.handle)
        .join(SourceChannel, SourceChannel.creator_id == Creator.id)
    ).all()
df_creators = pd.DataFrame(rows, columns=['creator', 'source', 'handle'])
df_creators

## Document counts per creator

In [ ]:
from sqlalchemy import func

with session_scope() as s:
    rows = s.execute(
        select(
            Creator.display_name.label('creator'),
            func.count(Document.id).label('n_docs'),
            func.count(TranscriptSegment.id).label('n_segments'),
        )
        .join(SourceChannel, SourceChannel.creator_id == Creator.id)
        .join(Document, Document.source_channel_id == SourceChannel.id, isouter=True)
        .join(TranscriptSegment, TranscriptSegment.document_id == Document.id, isouter=True)
        .group_by(Creator.display_name)
        .order_by(Creator.display_name)
    ).all()
pd.DataFrame(rows)

## Sample transcript segments (first creator)

In [ ]:
with session_scope() as s:
    doc = s.execute(
        select(Document).order_by(Document.posted_at.desc()).limit(1)
    ).scalar_one_or_none()
if doc is None:
    print('No documents ingested yet — run `tsr ingest`.')
else:
    print(f'{doc.title} ({doc.posted_at})')
    print(doc.url)
    with session_scope() as s:
        segs = s.execute(
            select(TranscriptSegment).where(TranscriptSegment.document_id == doc.id).limit(20)
        ).scalars().all()
    for seg in segs:
        print(f'  [{seg.start_seconds:6.1f}s] {seg.text}')